# 1. 2D floor planning with rotation: minimise the area

Lecture: *Week 11 - Floor Planning*.

*Original script: `5 Source Files/Python/model.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same minimum area (12).*

*Note: several layouts reach the minimum area, so the rotations and positions drawn may differ from the slides; the area is the same.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *
import numpy as np
from scipy.special import comb
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm

In [ ]:
# Data
w = np.array([4,3,6,7,3,2])
h = np.array([5,7,4,6,3,4])

In [ ]:
# w = np.array([4,3,6,7])
# h = np.array([5,7,4,7])

M = 9999

no_items = len(w)
no_binary = int(comb(no_items,2))

In [ ]:
# Model
model = ConcreteModel()

In [ ]:
# Sets
model.I = RangeSet(no_items)
model.J = RangeSet(no_items)

In [ ]:
# Parameters
model.w = Param(model.I, initialize=lambda model, i: w[i-1])
model.h = Param(model.I, initialize=lambda model, i: h[i-1])

In [ ]:
# Variables
model.x = Var(model.I, within=NonNegativeReals)
model.y = Var(model.I, within=NonNegativeReals)
model.v = Var(within=NonNegativeReals)
model.xij = Var(model.I, model.J, within=Binary)
model.yij = Var(model.I, model.J, within=Binary)
model.r = Var(model.I, within=Binary)

In [ ]:
# Objective
model.obj = Objective(expr=model.v, sense=minimize)

In [ ]:
# Constraints
def overlap_xij_rule(model, i, j):
    if i < j:
        return model.x[i] + (1-model.r[i])*model.w[i] + model.r[i]*model.h[i] <= model.x[j] + M*(model.xij[i,j]+model.yij[i,j])
    
    return Constraint.Skip
    
model.overlap_xij = Constraint(model.I, model.J, rule=overlap_xij_rule)

def overlap_xji_rule(model, i, j):
    if i < j:
        return model.x[j] + (1-model.r[j])*model.w[j] + model.r[j]*model.h[j] <= model.x[i] + M*(1-model.xij[i,j]+model.yij[i,j])
    
    return Constraint.Skip
model.overlap_xji = Constraint(model.I, model.J, rule=overlap_xji_rule)

def overlap_yij_rule(model, i, j):
    if i < j:
        return model.y[i] + (1-model.r[i])*model.h[i] + model.r[i]*model.w[i] <= model.y[j] + M*(1+model.xij[i,j]-model.yij[i,j])
        
    return Constraint.Skip

model.overlap_yij = Constraint(model.I, model.J, rule=overlap_yij_rule)

def overlap_yji_rule(model, i, j):
    if i < j:
        return model.y[j] + (1-model.r[j])*model.h[j] + model.r[j]*model.w[j] <= model.y[i] + M*(2-model.xij[i,j]-model.yij[i,j])
    
    return Constraint.Skip
model.overlap_yji = Constraint(model.I, model.J, rule=overlap_yji_rule)

def box_xi_rule(model, i):
    return model.x[i] + (1-model.r[i])*model.w[i] + model.r[i]*model.h[i] <= model.v
model.box_xi = Constraint(model.I, rule=box_xi_rule)

def box_yi_rule(model, i):
    return model.y[i] + (1-model.r[i])*model.h[i] + model.r[i]*model.w[i] <= model.v
model.box_yi = Constraint(model.I, rule=box_yi_rule)

In [ ]:
# used to switch off rotation
def rotation_off(model, i):
    return model.r[i] <= 0

In [ ]:
# uncomment this to switch off rotation
#model.rotation_off = Constraint(model.I, rule=rotation_off)

# Solve
solver = SolverFactory('appsi_highs')
solver.solve(model, tee=True)

model.display()
model.pprint()

In [ ]:
# Print the value of the variables at the optimum
for i in model.I:
    print("x[",i,"]=", model.x[i].value)
    print("y[",i,"]=", model.y[i].value)
    print("r[",i,"]=", model.r[i].value)
print("v=", model.v.value)

fig, ax = plt.subplots()

cmap = plt.get_cmap('Pastel1')  # Can be any colormap available in matplotlib

for i in range(1, no_items+1):
    x = model.x[i].value
    y = model.y[i].value
    wi = w[i-1]
    hi = h[i-1]
    color = cmap(i/no_items)  # Get the color from the colormap
    if model.r[i].value:
        print('rotated')
        rect = patches.Rectangle((x,y),hi,wi,linewidth=2,edgecolor='k',facecolor=color)
    else:
        print('not rotated')
        rect = patches.Rectangle((x,y),wi,hi,linewidth=2,edgecolor='k',facecolor=color)
    ax.add_patch(rect)

plt.grid(True)
plt.axis('equal')
plt.legend([str(x) for x in range(1, no_items+1)], loc='lower left')
plt.xlabel('x')
plt.ylabel('y')
plt.show()